In [2]:
!pip install -q -U transformers accelerate huggingface_hub scipy

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = "/workspace/hf_cache"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
login(token="YOUR TOKEN")

MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_NAME = "llama_3.1_8b_fp16"
N_LAYERS   = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0})
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
torch.cuda.empty_cache()

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"dtype: {next(model.parameters()).dtype}")
print(f"layers: {model.config.num_hidden_layers}")
assert model.config.num_hidden_layers == N_LAYERS, "LAYER MISMATCH"
assert next(model.parameters()).dtype == torch.float16, "NOT fp16"
print("checks passed")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

GPU: NVIDIA A40
GPU memory: 16.06 GB
dtype: torch.float16
layers: 32
checks passed


In [6]:
!pip install -q pandas

In [7]:
import os, json, subprocess
import numpy as np

if not os.path.exists("/workspace/RuleArena"):
    subprocess.run(["git","clone","-q","https://github.com/SkyRiver-2000/RuleArena.git","/workspace/RuleArena"], check=True)

os.chdir("/workspace/RuleArena/airline")
exec(open("compute_answer.py").read().split("if __name__")[0])
check_base_tables = load_checking_fee()

problems = []
with open("synthesized_problems/comp_0.jsonl") as f:
    for line in f:
        problems.append(json.loads(line))

def compute_gt(info):
    total, gt_info = compute_answer(
        base_price=info["base_price"], direction=info["direction"],
        routine=info["routine"], customer_class=info["customer_class"],
        bag_list=[{"id":0,"name":"ticket","size":[0,0,0],"weight":0}] + info["bag_list"],
        check_base_tables=check_base_tables)
    return total, gt_info

print("Problems:", len(problems), "| GT problem 0:", compute_gt(problems[0]["info"])[0])

Problems: 100 | GT problem 0: 1445


In [8]:
import re

with open("/workspace/RuleArena/airline/reference_rules.txt") as f:
    REFERENCE_RULES = f.read()
print(f"Rules loaded: {len(REFERENCE_RULES)} chars")


def generate_steps(problem, gt_info):
    info = problem["info"]
    steps = []

    for i, bag in enumerate(info["bag_list"]):
        bag_id = bag["id"]
        dims = bag["size"]
        weight = bag["weight"]
        dim_sum = sum(dims)

        correct_oversize_fee = gt_info["oversize"][i]
        correct_overweight_fee = gt_info["overweight"][i]
        correct_base_fee = gt_info["base"][i]

        steps.append({
            "step_id": f"bag{bag_id}_EA",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compute dimension sum",
            "system": "You are a calculator. Answer with just the number.",
            "user": f"What is {dims[0]} + {dims[1]} + {dims[2]}? Answer with just the number.",
            "correct_answer": str(dim_sum),
        })

        expected_oversize = "OVERSIZE" if dim_sum > 62 else "NOT OVERSIZE"
        steps.append({
            "step_id": f"bag{bag_id}_EB",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare sum to oversize threshold",
            "system": "Answer with exactly OVERSIZE or NOT OVERSIZE.",
            "user": (
                f"A bag has total dimensions of {dim_sum} inches. "
                f"The oversize threshold is 62 inches. "
                f"Is this bag OVERSIZE or NOT OVERSIZE?"
            ),
            "correct_answer": expected_oversize,
        })

        expected_overweight = "OVERWEIGHT" if weight > 50 else "NOT OVERWEIGHT"
        steps.append({
            "step_id": f"bag{bag_id}_EC",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare weight to overweight threshold",
            "system": "Answer with exactly OVERWEIGHT or NOT OVERWEIGHT.",
            "user": (
                f"A bag weighs {weight} lbs. "
                f"The overweight threshold is 50 lbs. "
                f"Is this bag OVERWEIGHT or NOT OVERWEIGHT?"
            ),
            "correct_answer": expected_overweight,
        })

        if dim_sum > 62:
            if dim_sum <= 65:
                bracket = "Over 62 inches to 65 inches"
            else:
                bracket = "Over 65 inches to 115 inches"
            steps.append({
                "step_id": f"bag{bag_id}_ED",
                "bag_id": bag_id,
                "type": "elementary" if correct_oversize_fee > 0 else "boundary",
                "description": "Read oversize fee from table given bracket"
                               + (" (complimentary rule applies -> boundary)" if correct_oversize_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {bracket}.\n\n"
                    f"OVERSIZE FEE TABLE:\n"
                    f"Between U.S., Puerto Rico, U.S. Virgin Islands and Canada:\n"
                    f"- Over 62 inches to 65 inches: $30\n"
                    f"- Over 65 inches to 115 inches: $200\n\n"
                    f"What is the oversize fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            if weight <= 53:
                wbracket = "Over 50 lbs to 53 lbs"
            elif weight <= 70:
                wbracket = "Over 53 lbs to 70 lbs"
            else:
                wbracket = "Over 70 lbs to 100 lbs"
            steps.append({
                "step_id": f"bag{bag_id}_EE",
                "bag_id": bag_id,
                "type": "elementary" if correct_overweight_fee > 0 else "boundary",
                "description": "Read overweight fee from table given bracket"
                               + (" (complimentary rule applies -> boundary)" if correct_overweight_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {wbracket}.\n\n"
                    f"OVERWEIGHT FEE TABLE:\n"
                    f"- Over 50 lbs to 53 lbs: $30\n"
                    f"- Over 53 lbs to 70 lbs: $100\n"
                    f"- Over 70 lbs to 100 lbs: $200\n\n"
                    f"What is the overweight fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

        steps.append({
            "step_id": f"bag{bag_id}_B0",
            "bag_id": bag_id,
            "type": "boundary",
            "description": "Is bag complimentary? (class + bag number + route rule)",
            "system": "Answer with exactly YES or NO.",
            "user": (
                f"Passenger class: {gt_info['customer_class']}\n"
                f"Route: {gt_info['routine']} domestic\n"
                f"This is bag number {bag_id} "
                f"(the {['1st','2nd','3rd','4th','5th'][bag_id-1]} checked bag).\n\n"
                f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                f"Is this bag complimentary (free) for this passenger? "
                f"Answer with exactly YES or NO."
            ),
            "correct_answer": "YES" if correct_base_fee == 0 else "NO",
        })

        if dim_sum > 62:
            steps.append({
                "step_id": f"bag{bag_id}_B1",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw dimensions -> oversize fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag dimensions: {dims[0]} x {dims[1]} x {dims[2]} inches.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the oversize fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            steps.append({
                "step_id": f"bag{bag_id}_B2",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw weight -> overweight fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag weight: {weight} lbs.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the overweight fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

    return steps


def extract_answer(response, step_type):
    response = response.strip()
    for keyword in ["NOT OVERSIZE", "NOT OVERWEIGHT", "OVERSIZE", "OVERWEIGHT"]:
        if keyword in response.upper():
            return keyword
    if response.upper().startswith("YES"):
        return "YES"
    if response.upper().startswith("NO"):
        return "NO"
    match = re.search(r'\$(\d+)', response)
    if match:
        return f"${match.group(1)}"
    nums = re.findall(r'\b\d+\b', response)
    if nums:
        return nums[0]
    return response.strip()


def build_prompt(step):
    messages = [
        {"role": "system", "content": step["system"]},
        {"role": "user", "content": step["user"]},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


gt, gt_info = compute_gt(problems[0]["info"])
_steps = generate_steps(problems[0], gt_info)
print(f"Problem 0: {len(_steps)} steps - "
      f"{sum(1 for s in _steps if s['type']=='elementary')} elementary, "
      f"{sum(1 for s in _steps if s['type']=='boundary')} boundary")

Rules loaded: 19583 chars
Problem 0: 36 steps - 23 elementary, 13 boundary


In [9]:
import torch
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

@torch.no_grad()
def answer_and_lens_fast(step, max_new_tokens=8):
    prompt = build_prompt(step)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = enc["input_ids"].shape[-1]

    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        output_hidden_states=True, return_dict_in_generate=True,
    )
    answer_ids = out.sequences[0][prompt_len:]
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)

    ans_tokens = [tokenizer.decode([t]) for t in answer_ids]
    read_off = 0
    for j, t in enumerate(ans_tokens):
        if any(ch.isdigit() for ch in t):
            read_off = j
            break

    target_id = answer_ids[read_off].item()
    step_hs = out.hidden_states[read_off]
    norm, head = model.model.norm, model.lm_head

    commit_top1, commit_p90, probs = None, None, []
    for L in range(1, N_LAYERS + 1):
        h = step_hs[L][0, -1]
        p = torch.softmax(head(norm(h)).float(), dim=-1)
        pt = p[target_id].item()
        probs.append(round(pt, 4))
        if commit_top1 is None and torch.argmax(p).item() == target_id:
            commit_top1 = L
        if commit_p90 is None and pt > 0.9:
            commit_p90 = L
        if L == N_LAYERS:
            v, i = torch.topk(p, 5)
            top5 = [(tokenizer.decode([a]), round(float(b), 4)) for a, b in zip(i.tolist(), v.tolist())]

    return {
        "answer_text": answer_text,
        "readout_token": tokenizer.decode([target_id]),
        "commit_top1": commit_top1 or N_LAYERS,
        "commit_p90": commit_p90 or N_LAYERS,
        "probs_by_layer": probs,
        "top5_final": top5,
    }

print("lens ready")

lens ready


In [10]:
import time
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)

test = []
for s in steps:
    r = answer_and_lens_fast(s)
    pred = extract_answer(r["answer_text"], s["type"])
    ok = pred.strip().lstrip("$").upper() == s["correct_answer"].strip().lstrip("$").upper()
    test.append({"step_id": s["step_id"], "correct": ok, "commit": r["commit_top1"],
                 "readout": r["readout_token"], "top5": r["top5_final"]})

def sfx(x): return x["step_id"].split("_")[1]
giv = [x for x in test if sfx(x) in ("ED","EE")]
der = [x for x in test if sfx(x) in ("B1","B2")]
mins = (time.time()-t0)/60
print(f"steps={len(test)}  time={mins:.1f} min")
print(f"PEAK GPU MEMORY: {torch.cuda.max_memory_allocated()/1e9:.1f} GB of 48")
print(f"GIVEN   n={len(giv)} acc={np.mean([x['correct'] for x in giv]):.0%} L={np.mean([x['commit'] for x in giv]):.2f}")
print(f"DERIVED n={len(der)} acc={np.mean([x['correct'] for x in der]):.0%} L={np.mean([x['commit'] for x in der]):.2f}")
print("readouts:", [x["readout"] for x in test[:6]])
print(f"ETA 60 problems: {mins*60:.0f} min  =  ${mins*60/60*0.45:.2f}")

steps=36  time=0.2 min
PEAK GPU MEMORY: 17.9 GB of 48
GIVEN   n=8 acc=100% L=24.00
DERIVED n=8 acc=38% L=23.00
readouts: ['41', 'NOT', 'NOT', 'NO', '86', 'OVER']
ETA 60 problems: 15 min  =  $0.11


In [11]:
import time, json

START_IDX = 0
END_IDX = 60
RESULTS_PATH = "/workspace/results_sweep_llama_fp16.json"

if os.path.exists(RESULTS_PATH):
    all_results = json.load(open(RESULTS_PATH))
    done = {r["problem_idx"] for r in all_results}
    print(f"Resuming - {len(done)} problems.")
else:
    all_results, done = [], set()
    print("Starting fresh.")

t_start = time.time()

for idx in range(START_IDX, END_IDX):
    if idx in done: continue
    try:
        gt, gt_info = compute_gt(problems[idx]["info"])
        steps = generate_steps(problems[idx], gt_info)
        for s in steps:
            r = answer_and_lens_fast(s)
            pred = extract_answer(r["answer_text"], s["type"])
            all_results.append({
                "problem_idx": idx, "step_id": s["step_id"], "type": s["type"],
                "description": s["description"], "correct_answer": s["correct_answer"],
                "predicted": pred,
                "correct": pred.strip().lstrip("$").upper() == s["correct_answer"].strip().lstrip("$").upper(),
                "readout_token": r["readout_token"],
                "commit_top1": r["commit_top1"], "commit_p90": r["commit_p90"],
                "probs_by_layer": r["probs_by_layer"], "top5_final": r["top5_final"],
            })
        with open(RESULTS_PATH,"w") as f: json.dump(all_results,f)
        def sfx(x): return x["step_id"].split("_")[1]
        giv=[x for x in all_results if sfx(x) in ("ED","EE")]
        der=[x for x in all_results if sfx(x) in ("B1","B2")]
        mins=(time.time()-t_start)/60
        print(f"[{idx+1}/60] steps={len(all_results)}  "
              f"GIVEN acc={np.mean([x['correct'] for x in giv]):.1%} L={np.mean([x['commit_top1'] for x in giv]):.2f}  |  "
              f"DERIVED acc={np.mean([x['correct'] for x in der]):.1%} L={np.mean([x['commit_top1'] for x in der]):.2f}  ({mins:.1f}m)")
    except Exception as e:
        print(f"  ERROR {idx}: {e}")
        torch.cuda.empty_cache()
        continue

print(f"\nDone. {len(all_results)} steps, {(time.time()-t_start)/60:.1f} min.")

Starting fresh.
[1/60] steps=36  GIVEN acc=100.0% L=24.00  |  DERIVED acc=37.5% L=23.00  (0.2m)
[2/60] steps=72  GIVEN acc=93.8% L=24.12  |  DERIVED acc=31.2% L=23.00  (0.5m)
[3/60] steps=108  GIVEN acc=91.7% L=24.00  |  DERIVED acc=37.5% L=23.00  (0.7m)
[4/60] steps=144  GIVEN acc=93.8% L=24.06  |  DERIVED acc=37.5% L=23.00  (0.9m)
[5/60] steps=180  GIVEN acc=85.0% L=24.20  |  DERIVED acc=30.0% L=23.02  (1.2m)
[6/60] steps=216  GIVEN acc=87.5% L=24.08  |  DERIVED acc=35.4% L=23.02  (1.4m)
[7/60] steps=252  GIVEN acc=85.7% L=24.00  |  DERIVED acc=37.5% L=23.02  (1.6m)
[8/60] steps=288  GIVEN acc=84.4% L=23.88  |  DERIVED acc=42.2% L=23.02  (1.9m)
[9/60] steps=324  GIVEN acc=86.1% L=23.86  |  DERIVED acc=43.1% L=23.03  (2.1m)
[10/60] steps=360  GIVEN acc=87.5% L=23.88  |  DERIVED acc=42.5% L=23.02  (2.3m)
[11/60] steps=396  GIVEN acc=88.6% L=23.89  |  DERIVED acc=42.0% L=23.02  (2.6m)
[12/60] steps=432  GIVEN acc=89.6% L=23.90  |  DERIVED acc=42.7% L=23.03  (2.8m)
[13/60] steps=468  GIV